# BVI-Net -- improvements run (GPU T4, Internet on)

1. GPU FPS benchmark with the batched 4-direction scan (compare: 47 / 46 / 30 FPS before).
2. **Ablations on ISIC2018** (same split as the main skin model): without the Gabor bank,
   without the FA-VSSM global pathway, without the GCN skips.
3. **Liver fine-tune on 2.5x more data**: every 2nd labelled slice, 3x tumor oversampling,
   starting from the shipped liver checkpoint.

Attach: `tschandl/isic2018-challenge-task1-data-segmentation`, `andrewmvd/liver-tumor-segmentation`,
`andrewmvd/liver-tumor-segmentation-part-2`.

In [ ]:
!pip install causal-conv1d mamba-ssm --no-build-isolation

In [ ]:
import os
if os.path.basename(os.getcwd()) != "BVI-Net-ISIC":
    !git clone https://github.com/Ghanasree-S/BVI-Net-ISIC.git
    %cd BVI-Net-ISIC
!pip install -q -r requirements.txt
from models.fa_vssm import HAS_MAMBA
print("Real Mamba:", HAS_MAMBA)
os.makedirs("outputs", exist_ok=True)

## 1. GPU benchmark (batched scan)

In [ ]:
!python benchmark.py --iters 200 --out outputs/benchmark_batched.json

## 2. Ablations (ISIC2018)

In [ ]:
!bash data/download_isic_official_val.sh
!python data/prepare_isic_a.py --raw_dir /kaggle/input --val_raw_dir data/isic2018_val_raw --out_dir data/isic2018a --fraction 1.0

In [ ]:
for ab in ["gabor", "global", "gcn"]:
    print(f"===== ablation: without {ab} =====", flush=True)
    !python train.py --dataset isic --data_dir data/isic2018a --epochs 50 --batch_size 8 --lr 0.001 --patience 15         --channels 4 8 8 16 16 --gcn_nodes 8 --ablate {ab} --out_dir checkpoints/ablate_{ab}
    !python evaluate.py --dataset isic --checkpoint checkpoints/ablate_{ab}/best.pt --data_dir data/isic2018a         --channels 4 8 8 16 16 --gcn_nodes 8 --ablate {ab} --out_dir outputs/ablate_{ab}

## 3. Liver fine-tune on every 2nd slice

In [ ]:
!python data/prepare_lits.py --raw_dir /kaggle/input --out_dir data/lits17b --slice_stride 2
!python train.py --dataset lits --data_dir data/lits17b --epochs 15 --batch_size 4 --lr 0.0003 --patience 6     --channels 4 8 8 16 16 --gcn_nodes 8 --out_dir checkpoints/liver2     --init_checkpoint checkpoints/bvi_net_liver_best.pt --oversample_class 1 --oversample_factor 3
# Evaluate on the SAME stride-5 test slices as the shipped model, so the numbers are comparable.
!python data/prepare_lits.py --raw_dir /kaggle/input --out_dir data/lits17a --slice_stride 5
!python evaluate.py --dataset lits --checkpoint checkpoints/liver2/best.pt --data_dir data/lits17a     --channels 4 8 8 16 16 --gcn_nodes 8 --out_dir outputs/liver2

## 4. Package results

In [ ]:
import zipfile, glob
with zipfile.ZipFile("/kaggle/working/improvements.zip", "w") as z:
    for f in glob.glob("outputs/*.json") + glob.glob("outputs/*/metrics.json"):
        z.write(f, arcname=f)
    if os.path.exists("checkpoints/liver2/best.pt"):
        z.write("checkpoints/liver2/best.pt", arcname="bvi_net_liver_stride2.pt")
    for ab in ["gabor", "global", "gcn"]:
        p = f"checkpoints/ablate_{ab}/best.pt"
        if os.path.exists(p): z.write(p, arcname=f"bvi_net_skin_ablate_{ab}.pt")
print(zipfile.ZipFile("/kaggle/working/improvements.zip").namelist())